In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils import prune
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.utils.data as data
from torch.utils.data import Subset
import torch_pruning as tp
from ptflops import get_model_complexity_info
from ultralytics import YOLO
import os

from yolo_test import test_yolo_model

## Helper Functions

In [7]:
# Calculate FLOPs and parameters (adjusted for 1-channel input)
def print_model_flops(model, input_res=(1, 3, 32, 32), name="Model"):
    macs, params = get_model_complexity_info(model, input_res, as_strings=True,
                                             print_per_layer_stat=False, verbose=False)
    print(f"{name} – FLOPs: {macs}, Parameters: {params}")


In [ ]:
def prune_yolo_model(model: YOLO, prune_amount: float, save_path: str):
    """
    对 YOLO 模型进行剪枝并保存。

    Args:
        model (YOLO): 预加载的 YOLO 模型对象 (e.g., YOLO('yolov11m.pt')).
        prune_amount (float): 剪枝的比例，介于 0.0 到 1.0 之间 (e.g., 0.5 表示剪枝 50%).
        save_path (str): 剪枝后模型的保存路径 (e.g., 'yolov11m_pruned.pt').
    """
    if not (0.0 <= prune_amount <= 1.0):
        raise ValueError("prune_amount 必须在 0.0 到 1.0 之间。")

    print(f"--- 开始对 YOLO 模型进行剪枝 (剪枝比例: {prune_amount*100:.2f}%) ---")

    # 1. 应用剪枝
    # 遍历模型的所有模块并对 Conv2d 层进行剪枝
    pruned_layers_count = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            try:
                prune.l1_unstructured(module, name='weight', amount=prune_amount)
                # print(f"  已剪枝层: {name}") # 可以取消注释查看每个剪枝的层
                pruned_layers_count += 1
            except Exception as e:
                print(f"  警告: 无法剪枝层 {name}: {e}")

    if pruned_layers_count == 0:
        print("未发现或成功剪枝任何 Conv2d 层。请检查模型架构。")
        return # 如果没有剪枝，直接退出

    print(f"成功对 {pruned_layers_count} 个 Conv2d 层应用了剪枝。")

    # 2. 移除剪枝的重新参数化（使剪枝永久化）
    print("正在移除剪枝的重新参数化，使其永久化...")
    removed_layers_count = 0
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Conv2d):
            # 检查该模块是否已被剪枝，通过检查是否有 weight_orig 属性
            if hasattr(module, 'weight_orig'):
                prune.remove(module, 'weight')
                # print(f"  已移除层 {name} 的重新参数化") # 可以取消注释查看每个移除的层
                removed_layers_count += 1
    
    if removed_layers_count == 0:
        print("没有层需要移除重新参数化。这可能表示剪枝没有成功应用。")
    else:
        print(f"成功移除了 {removed_layers_count} 个层的重新参数化。")

    flops, params = profile(model, inputs=(example_inputs,))
    print(f"FLOPs: {flops / 1e6:.2f}M, Params: {params / 1e6:.2f}M")

    # 3. 保存剪枝后的模型
    try:
        # model.save(save_path)
        torch.save()
        print(f"剪枝后的模型已成功保存到: {save_path}")
    except Exception as e:
        print(f"保存剪枝模型时发生错误: {e}")
        # 如果保存失败，尝试删除可能已创建的部分文件
        if os.path.exists(save_path):
            os.remove(save_path)

    print("--- 剪枝过程完成 ---")


## Training and Evaluation

In [15]:
# Prepare model
model = YOLO("runs/detect/train3/weights/best.pt").model
data_yml = "train.yaml"

device = "cuda:0"
# FLOPs & Accuracy before pruning
print_model_flops(model, name="Before Pruning")

# Example input for pruning
example_inputs = torch.randn(1, 1, 28, 28).to(device)

# Pruning setup
importance = tp.importance.MagnitudeImportance(p=1)  # L1 norm

# AGP setup: start at 10%, end at 50% sparsity over 3 steps
num_pruning_steps = 3
epochs_per_step = 5
initial_ratio = 0.1
final_ratio = 0.5
ratios = torch.linspace(initial_ratio, final_ratio, steps=num_pruning_steps)

for step, ratio in enumerate(ratios):
    print(f"=== Pruning Step {step+1}/{num_pruning_steps} (Ratio: {ratio:.2f}) ===")

    pruned_save_path = f"prunes/save{step}.pt"
    prune_model(model, ratio, pruned_save_path)

    pruned_model = YOLO(pruned_save_path)

    # train_epochs(model, device, trainloader, epochs=epochs_per_step)
    train_results = pruned_model.train(
        data=data_yml,  # Path to dataset YAML (must be segmentation-compatible)
        epochs=epochs_per_step,  # Number of training epochs
        imgsz=32,  # Image size
        device=0,  # GPUs to use (or "cpu" for CPU training)
        batch=320,  # Adjust batch size based on GPU memory
        workers=4,  # Number of dataloader workers
        optimizer="AdamW",  # AdamW optimizer (optional, can use "SGD")
        lr0=0.01,  # Initial learning rate
        lrf=0.01,
        patience=50,  # Early stopping patience
        seed=42,  # Random seed for reproducibility
        verbose=True, # Display training progress
        multi_scale=False,
        pretrained = True,
        single_cls = False,
        cos_lr=True,
        box = 15,
    )
    print_model_flops(prune_model.model, name=f"After Pruning Step {step+1}")
    # evaluate_model(model, testloader, device, name=f"After Pruning Step {step+1}")
    test_yolo_model(pruned_save_path, data_yml, 32, 320, device=device)


print("Cov Pruning (AGP) + Fine-Tuning completed!")


Flops estimation was not finished successfully because of the following exception:
<class 'RuntimeError'> : Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [1, 1, 3, 32, 32]
Before Pruning – FLOPs: None, Parameters: None
=== Pruning Step 1/3 (Ratio: 0.10) ===


Traceback (most recent call last):
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ptflops/pytorch_engine.py", line 64, in get_flops_pytorch
    _ = flops_model(batch)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1603, in _call_impl
    result = forward_call(*args, **kwargs)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/nn/tasks.py", line 115, in forward
    return self.predict(x, *args, **kwargs)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/nn/tasks.py", line 133, in predict
    return self._predict_once(x, profile, visualize, embed)
  File "/home/supercomputing/miniconda3/envs/acv/lib/python3.8/site-packages/ultralytics/nn/ta

AttributeError: module 'torch_pruning' has no attribute 'DependencyAnalyzer'